# Module 4: Advanced RAG with Vector Databases and Retrievers
## Topics Covered
- Advanced Retrievers & Retrieval Patterns
- Retrieval Optimization
- FAISS: Efficient Similarity Search & Vector Indexing
- RAG Application Development: FAISS + LangChain + Gradio Integration

---
**Real-world scenario**: Building an advanced technical documentation search system for a software product with multi-strategy retrieval, re-ranking, and a Gradio UI.

In [ ]:
%pip install -q faiss-cpu langchain langchain-openai langchain-community
%pip install -q gradio rank-bm25 tiktoken

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-api-key-here")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("Ready:", OPENAI_API_KEY != "your-api-key-here")

---
## 1. FAISS: Facebook AI Similarity Search

**FAISS** is a library for efficient similarity search at billion-vector scale.

```
FAISS Index Types:
├── IndexFlatL2       → Exact search, small datasets (<100k)
├── IndexFlatIP       → Exact search with inner product
├── IndexIVFFlat      → Approximate search with inverted file
├── IndexIVFPQ        → Compressed, billion-scale
└── IndexHNSWFlat     → Graph-based, best accuracy/speed tradeoff
```

In [ ]:
import faiss
import numpy as np
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Sample technical documentation chunks
tech_docs = [
    {"id": 0, "text": "API Authentication: Use Bearer tokens in the Authorization header. Tokens expire after 24 hours. Refresh using the /auth/refresh endpoint.", "section": "auth"},
    {"id": 1, "text": "Rate Limiting: API calls are limited to 1000 requests/hour per API key. Exceeding this returns HTTP 429. Use exponential backoff for retries.", "section": "limits"},
    {"id": 2, "text": "Webhook Configuration: Register webhook URLs via POST /webhooks. Events include: payment.completed, subscription.created, user.deleted.", "section": "webhooks"},
    {"id": 3, "text": "Error Codes: 400 Bad Request - invalid parameters. 401 Unauthorized - invalid API key. 403 Forbidden - insufficient permissions. 404 Not Found.", "section": "errors"},
    {"id": 4, "text": "Pagination: All list endpoints support cursor-based pagination. Use 'next_cursor' from response to fetch next page. Default page size is 20.", "section": "pagination"},
    {"id": 5, "text": "Data Formats: All requests and responses use JSON. Dates are ISO 8601 format (YYYY-MM-DDTHH:MM:SSZ). Monetary values in smallest currency unit.", "section": "formats"},
    {"id": 6, "text": "SDK Installation: pip install acme-sdk. Initialize with AcmeClient(api_key='your-key'). Python 3.8+ required. Async support available.", "section": "sdk"},
    {"id": 7, "text": "Subscription Management: Create subscriptions via POST /subscriptions. Update with PATCH. Cancel with DELETE. Proration is automatic on plan changes.", "section": "subscriptions"},
    {"id": 8, "text": "Sandbox Environment: Test using sandbox.acme.com with test API keys prefixed 'test_'. Test credit cards: 4242424242424242 always succeeds.", "section": "testing"},
    {"id": 9, "text": "Token Refresh: POST /auth/refresh with refresh_token. New access token valid for 24 hours. Refresh tokens valid for 30 days. Store securely.", "section": "auth"}
]

# Get embeddings
texts = [d["text"] for d in tech_docs]
doc_embeddings = embeddings_model.embed_documents(texts)
doc_matrix = np.array(doc_embeddings, dtype='float32')

print(f"Documents: {len(tech_docs)} | Dimensions: {doc_matrix.shape[1]}")

In [ ]:
# Build different FAISS indices
dim = doc_matrix.shape[1]

# 1. Exact search index (L2 distance)
index_flat = faiss.IndexFlatL2(dim)
index_flat.add(doc_matrix)

# 2. Inner Product index (for cosine similarity with normalized vectors)
# Normalize first for cosine similarity
doc_matrix_normalized = doc_matrix.copy()
faiss.normalize_L2(doc_matrix_normalized)
index_ip = faiss.IndexFlatIP(dim)
index_ip.add(doc_matrix_normalized)

# 3. HNSW index (best for production - graph-based ANN)
index_hnsw = faiss.IndexHNSWFlat(dim, 32)  # 32 = connections per node
index_hnsw.add(doc_matrix)

print(f"Flat L2 index: {index_flat.ntotal} vectors")
print(f"Inner Product index: {index_ip.ntotal} vectors")
print(f"HNSW index: {index_hnsw.ntotal} vectors")

def search_index(query, index, k=3, normalize=False):
    """Search FAISS index and return top-k results"""
    query_vec = np.array([embeddings_model.embed_query(query)], dtype='float32')
    if normalize:
        faiss.normalize_L2(query_vec)
    distances, indices = index.search(query_vec, k)
    return distances[0], indices[0]

# Test search
query = "How do I authenticate API requests?"
dists, idxs = search_index(query, index_flat, k=3)

print(f"\n🔍 Query: '{query}'")
for dist, idx in zip(dists, idxs):
    if idx >= 0:
        print(f"\n[Score: {dist:.4f}] {tech_docs[idx]['section'].upper()}:")
        print(f"  {tech_docs[idx]['text'][:100]}...")

---
## 2. Advanced Retrieval Patterns

### Retrieval Strategies:
- **Dense Retrieval**: Semantic vector search
- **Sparse Retrieval (BM25)**: Keyword-based TF-IDF style
- **Hybrid Retrieval**: Combine dense + sparse (best of both)
- **Multi-Query**: Generate multiple queries to increase recall
- **Contextual Compression**: Extract only relevant parts

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.schema import Document

# Build LangChain FAISS vectorstore
documents = [Document(page_content=d["text"], metadata={"section": d["section"], "id": d["id"]}) 
             for d in tech_docs]

vectorstore = FAISS.from_documents(documents, embeddings_model)
print("FAISS vectorstore created with", vectorstore.index.ntotal, "vectors")

In [ ]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever

# BM25 (sparse) retriever - good for exact keyword matching
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 3

# FAISS (dense) retriever - good for semantic matching
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Hybrid: Ensemble of BM25 + FAISS (Reciprocal Rank Fusion)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.4, 0.6]  # 40% sparse, 60% dense
)

# Compare retrieval methods
test_query = "token expiration and refresh"

print(f"🔍 Query: '{test_query}'\n")

print("--- BM25 (Keyword) Results ---")
for doc in bm25_retriever.invoke(test_query):
    print(f"  [{doc.metadata['section']}] {doc.page_content[:80]}...")

print("\n--- FAISS (Semantic) Results ---")
for doc in faiss_retriever.invoke(test_query):
    print(f"  [{doc.metadata['section']}] {doc.page_content[:80]}...")

print("\n--- Hybrid Ensemble Results ---")
for doc in ensemble_retriever.invoke(test_query):
    print(f"  [{doc.metadata['section']}] {doc.page_content[:80]}...")

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Multi-Query Retriever: generates multiple query variations to improve recall
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=faiss_retriever,
    llm=llm
)

import logging
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

question = "I keep getting 401 errors, how do I fix this?"
print(f"Query: '{question}'")
print("(Multi-query generates variations to improve retrieval recall)\n")

unique_docs = multi_query_retriever.invoke(question)
print(f"\nRetrieved {len(unique_docs)} unique documents:")
for doc in unique_docs:
    print(f"  [{doc.metadata['section']}] {doc.page_content[:80]}...")

---
## 3. Contextual Compression Retriever

Extracts only the **relevant portions** of retrieved documents — reduces noise in context.

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

# Compressor: extracts relevant sentences from retrieved docs
compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=faiss_retriever
)

query = "What HTTP status code means I'm being throttled?"
compressed_docs = compression_retriever.invoke(query)

print(f"Query: '{query}'")
print("\nCompressed (relevant portions only):")
for doc in compressed_docs:
    print(f"  ✂️ {doc.page_content}")

---
## 4. Complete RAG App: FAISS + LangChain + Gradio

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import gradio as gr

# RAG prompt with source attribution
rag_prompt = ChatPromptTemplate.from_template("""
You are a technical documentation assistant. Answer based on the provided documentation context.
Be specific and include relevant details like endpoints, status codes, or parameters.
If you're uncertain, say so.

Documentation context:
{context}

Question: {question}

Answer:
""")

def format_docs_with_sources(docs):
    return "\n\n".join(
        f"[{doc.metadata['section'].upper()}]: {doc.page_content}" 
        for doc in docs
    )

# Full RAG chain with hybrid retrieval
rag_chain = (
    {"context": ensemble_retriever | format_docs_with_sources, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

def technical_doc_chat(message, history):
    """Technical docs chatbot with FAISS + hybrid retrieval"""
    try:
        # Get relevant docs for transparency
        retrieved = ensemble_retriever.invoke(message)
        sections = list(set([doc.metadata['section'] for doc in retrieved]))
        
        answer = rag_chain.invoke(message)
        
        return f"{answer}\n\n📚 *Sections consulted: {', '.join(sections)}*"
    except Exception as e:
        return f"Error: {str(e)}"

# Gradio UI
with gr.Blocks(theme=gr.themes.Base()) as app:
    gr.Markdown("# 📖 Technical Documentation Assistant")
    gr.Markdown("*Powered by FAISS + Hybrid Retrieval (BM25 + Dense) + GPT-4o-mini*")
    
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.ChatInterface(
                fn=technical_doc_chat,
                examples=[
                    "How do I authenticate my API requests?",
                    "I'm getting a 429 error, what should I do?",
                    "How does pagination work?",
                    "How do I test without affecting production?",
                    "My auth token expired, how do I refresh it?"
                ]
            )

app.launch(share=False, show_error=True)

---
## 5. FAISS Index Persistence

In [ ]:
import os

# Save FAISS index to disk
save_path = "./faiss_tech_docs"
vectorstore.save_local(save_path)
print(f"FAISS index saved to: {save_path}")
print(f"Files: {os.listdir(save_path)}")

# Load from disk (production use)
loaded_vectorstore = FAISS.load_local(
    save_path,
    embeddings=embeddings_model,
    allow_dangerous_deserialization=True  # Required flag in newer versions
)
print(f"\nLoaded vectorstore with {loaded_vectorstore.index.ntotal} vectors")

# Verify it works
test_results = loaded_vectorstore.similarity_search("rate limit", k=2)
print(f"Test search returned {len(test_results)} results ✅")

---
## Summary

| Pattern | When to Use | Performance |
|---------|-------------|-------------|
| Flat L2/IP | Small datasets, exact results | Slow at scale |
| HNSW | Production, accuracy matters | Fast, slight recall loss |
| IVF-PQ | Billion-scale, memory constrained | Fastest, lossy |
| BM25 | Keyword-heavy queries | Exact keyword match |
| Hybrid | Best overall retrieval | Best of both worlds |
| Multi-Query | Low recall queries | Improved recall |
| Compression | Noisy docs, large chunks | Cleaner context |

### Next Steps
- Module 5: Multimodal AI — extend RAG to images, audio, and video